In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

laps_table = f"{catalog}.f1_silver.lap_times"
pits_table = f"{catalog}.f1_silver.pit_stops"

laps_df = spark.table(laps_table)
pits_df = spark.table(pits_table)

In [0]:
pit_boundaries_df = pits_df.select(
    "race_id",
    "driver_id",
    "pit_lap"
)

laps_marked_df = (
    laps_df.alias("laps")
    .join(
        pit_boundaries_df.alias("pits"),
        (
            (F.col("laps.race_id") == F.col("pits.race_id"))
            & (F.col("laps.driver_id") == F.col("pits.driver_id"))
            & (F.col("laps.lap_number") == F.col("pits.pit_lap") + 1)
        ),
        "left"
    )
    .select(
        F.col("laps.*"),
        F.when(
            F.col("pits.pit_lap").isNotNull(),
            1
        ).otherwise(0).alias("new_stint")
    )
)

In [0]:
stint_window = (
    Window
    .partitionBy("race_id", "driver_id")
    .orderBy("lap_number")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

laps_with_stints_df = (
    laps_marked_df
    .withColumn(
        "stint_number",
        F.sum("new_stint").over(stint_window) + 1
    )
)

display(
    laps_with_stints_df
    .select(
        "race_id",
        "driver_id",
        "lap_number",
        "lap_time_ms",
        "stint_number"
    )
    .orderBy("driver_id", "lap_number")
)

In [0]:
lap_order_window = (
    Window
    .partitionBy("race_id", "driver_id", "stint_number")
    .orderBy("lap_number")
)

laps_enriched_df = (
    laps_with_stints_df
    .withColumn(
        "lap_in_stint",
        F.row_number().over(lap_order_window)
    )
)

In [0]:
stint_metrics_df = (
    laps_enriched_df
    .groupBy(
        "race_id",
        "driver_id",
        "stint_number"
    )
    .agg(
        F.min("lap_number").alias("stint_start_lap"),
        F.max("lap_number").alias("stint_end_lap"),
        F.count("*").alias("stint_laps"),
        F.avg("lap_time_ms").alias("average_lap_time_ms"),
        F.min("lap_time_ms").alias("best_lap_time_ms"),
        F.max("lap_time_ms").alias("slowest_lap_time_ms")
    )
    .withColumn(
        "tyre_compound",
        F.lit("UNKNOWN")
    )
)

display(
    stint_metrics_df
    .orderBy("driver_id", "stint_number")
)

In [0]:
gold_table = f"{catalog}.f1_gold.driver_stint_metrics"

(
    stint_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"Saved Gold table: {gold_table}")

In [0]:
display(
    spark.table(gold_table)
    .orderBy("race_id", "driver_id", "stint_number")
)